# Reference Data Creation

This notebook generates the **reference dataset** used for drift monitoring.
The reference represents the data distribution when the fraud model was considered stable.


## Objective

- Load the original fraud dataset
- Apply the **same preprocessing** used during model training
- Recreate the **validation split**
- Generate model predictions
- Save reference features and predictions for drift monitoring


In [9]:
#Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from pathlib import Path


## Load Raw Dataset

The original dataset is loaded **without any modification**.


In [2]:
df = pd.read_csv("../data/raw/creditcard.csv")
df.shape


(284807, 31)

## Separate Features and Target

The target column `Class` indicates whether a transaction is fraudulent.


In [3]:
X = df.drop(columns=["Class"])
y = df["Class"]

y.value_counts()


Class
0    284315
1       492
Name: count, dtype: int64

## Recreate Training Split

We recreate the **same split logic** used during model training.
The validation set will be used as the reference baseline.

To ensure reproducibility, we fix the random seed before splitting the dataset.

In [4]:
# Fix random seed for reproducibility
np.random.seed(42)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

X_val.shape

(42721, 30)

## Load Trained Model and Scaler

The trained fraud detection model and its preprocessing artifacts
are loaded to generate reference predictions.


In [5]:
model = joblib.load("../models/fraud_model.pkl")
scaler = joblib.load("../models/standard_scaler.pkl")

## Apply Preprocessing to Reference Data

Only the features used during scaler training are transformed.


In [6]:
scale_cols = ["Time", "Amount"]

X_val_processed = X_val.copy()
X_val_processed[scale_cols] = scaler.transform(X_val[scale_cols])

## Generate Reference Predictions


In [7]:
y_val_proba = model.predict_proba(X_val_processed)[:, 1]

## Save Reference Dataset

The reference features and predictions are saved for drift monitoring.


In [10]:
# Create reference folder if it doesn't exist
ref_path = Path("../data/reference")
ref_path.mkdir(parents=True, exist_ok=True)

# Save reference features
pd.DataFrame(X_val_processed, columns=X_val.columns).to_parquet(
    ref_path / "features.parquet"
)

# Save reference predictions
pd.DataFrame({"prediction": y_val_proba}).to_parquet(
    ref_path / "predictions.parquet"
)


## Reference Dataset Created Successfully

- Validation set selected as reference baseline
- Same preprocessing and random seed applied
- Reference features and predictions saved to disk

This notebook should only be re-run if the model or preprocessing changes.
